# Chapter 05: Decoding, evaluation, and resumable training


Training and generation use the same logits differently. Training scores
known targets with cross entropy; generation transforms logits into a
token choice. This chapter also treats checkpointing and configuration
selection as correctness problems rather than incidental plumbing.


    This notebook is an independently written practice companion. It
    does not reproduce the book's prose, figures, data, or solutions.
    Read the [Chinese chapter](https://github.com/skindhu/Build-A-Large-Language-Model-CN/blob/main/cn-Book/5.%E5%9C%A8%E6%97%A0%E6%A0%87%E8%AE%B0%E6%95%B0%E6%8D%AE%E9%9B%86%E4%B8%8A%E8%BF%9B%E8%A1%8C%E9%A2%84%E8%AE%AD%E7%BB%83.md) for the complete narrative
    and exercise statements, and use the
    [canonical MIT companion code](https://github.com/rasbt/LLMs-from-scratch)
    as an additional reference.

    **Prerequisites:** Python 3.10+ and PyTorch 2.2+. Every example is
    synthetic, deterministic, CPU-friendly, and makes no network call.

    **Workflow**

    1. Read the concept and trace the shapes by hand.
    2. Replace one `LLM_TODO` and its `NotImplementedError`.
    3. Run that exercise's check cell.
    4. Explain the result before moving to the next exercise.
    5. Compare with `solution.py` only after making a serious attempt.

In [ ]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass
from typing import Any

import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.optim import Optimizer


@dataclass(frozen=True)
class DecodingProfile:
    temperature: float
    top_k: int


@dataclass(frozen=True)
class GPTConfig:
    name: str
    embedding_dim: int
    num_layers: int
    num_heads: int
    nominal_parameters: int

## Exercise 5.1: Observe temperature sampling

    **Objective.** Scale logits, sample reproducibly, and compare token-frequency concentration.


Temperature rescales logits before softmax:
$p_i=\exp(z_i/\tau)/\sum_j\exp(z_j/\tau)$. For $0<\tau<1$, logit
differences are amplified and probability concentrates on the largest
logit. For $\tau>1$, differences shrink and the distribution flattens.

Sampling frequencies are random estimates of probabilities, so exact
counts should not be asserted. A seeded generator makes a demonstration
reproducible, while enough draws make the concentration trend visible.
Temperature zero is handled by greedy selection, not division by zero.


    ### Data-flow walkthrough


For logits `[0,1,2]`, compare 2,000 draws at temperatures 0.2 and 2.0.
The last token should dominate much more strongly in the cold sample.
Return a count for every vocabulary entry, including entries not drawn.


    ### Hints and common mistakes


Apply temperature to logits, not probabilities. Use replacement because
each generation step samples from the full vocabulary. Pass an explicit
`torch.Generator` instead of changing global RNG state.

In [ ]:
def sample_frequencies(
    logits: Tensor, temperature: float, draws: int, seed: int
) -> Tensor:
    # LLM_TODO_CH05_01_TEMPERATURE
    raise NotImplementedError(
        "sample seeded counts from temperature-scaled logits"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
logits = torch.tensor([0.0, 1.0, 2.0])
cold = sample_frequencies(logits, 0.2, 2_000, seed=4)
warm = sample_frequencies(logits, 2.0, 2_000, seed=4)
assert cold.sum() == warm.sum() == 2_000
assert cold[-1] > warm[-1]
print("Exercise 5.1 passed")

## Exercise 5.2: Choose a decoding profile

    **Objective.** Represent deterministic and creative decoding goals as explicit settings.


`top_k` limits sampling to the $k$ largest logits; temperature reshapes
probabilities within the retained set. A deterministic workflow can set
`top_k=1`, while exploratory generation can retain more candidates and
use a temperature above one.

These are task policies, not universal quality settings. A profile
should be named by intent so callers do not scatter unexplained numeric
constants throughout generation code.


    ### Data-flow walkthrough


Return a dataclass for `"deterministic"` and `"creative"`. The former
must retain one candidate. The latter should retain multiple candidates
and use a temperature above one in this toy policy.


    ### Hints and common mistakes


Reject unknown profile names instead of silently choosing a default.
Avoid claiming that one profile is always better; generation goals and
model calibration differ.

In [ ]:
def choose_decoding_profile(task: str) -> DecodingProfile:
    # LLM_TODO_CH05_02_PROFILES
    raise NotImplementedError("map a named intent to explicit settings")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
assert choose_decoding_profile("deterministic").top_k == 1
creative = choose_decoding_profile("creative")
assert creative.top_k > 1 and creative.temperature > 1.0
print("Exercise 5.2 passed")

## Exercise 5.3: Generate deterministically

    **Objective.** Append the unique top-1 token at each autoregressive step.


Autoregressive generation repeatedly evaluates the current context,
selects a next token, appends it, and uses the extended sequence on the
next iteration. With `top_k=1`, sampling and argmax are equivalent
because only one candidate survives.

The function here receives a callback returning next-token logits with
shape `[1,V]`. Keeping that contract small separates generation logic
from any particular model implementation.


    ### Data-flow walkthrough


The toy callback makes the successor of token $t$ equal to
$(t+1)\bmod 5`. Starting with `[0]` and generating four tokens should
yield `[0,1,2,3,4]`.


    ### Hints and common mistakes


Select from the final logit row and append a Python integer. Validate the
callback shape. Do not mutate the caller's prompt list unexpectedly;
begin with a copy.

In [ ]:
def generate_deterministic(
    next_logits: Callable[[Tensor], Tensor],
    prompt: list[int],
    max_new_tokens: int,
    top_k: int = 1,
) -> list[int]:
    # LLM_TODO_CH05_03_DETERMINISTIC
    raise NotImplementedError("append one top-ranked token per step")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
def toy_next(context: Tensor) -> Tensor:
    target = (int(context[0, -1]) + 1) % 5
    result = torch.zeros(1, 5)
    result[0, target] = 10.0
    return result


prompt = [0]
generated = generate_deterministic(toy_next, prompt, 4)
assert generated == [0, 1, 2, 3, 4]
assert prompt == [0]
print("Exercise 5.3 passed")

## Exercise 5.4: Checkpoint and resume optimizer state

    **Objective.** Save model parameters, optimizer dynamics, and progress as one logical checkpoint.


Resuming training requires more than model weights. Optimizers can carry
momentum, running averages, and step counters that affect the next
update. A useful checkpoint stores both state dictionaries plus explicit
progress such as the completed step.

State dictionaries contain tensors that may later mutate, so an
in-memory snapshot should be deep-copied. Loading must target compatible
model and optimizer structures. In production, also preserve scheduler,
scaler, RNG, and data-order state when exact reproducibility matters.


    ### Data-flow walkthrough


Train a one-weight linear model for one step, snapshot it, restore into
a fresh model and momentum optimizer, then take the second step. Its
weight should match a model trained for two uninterrupted steps.


    ### Hints and common mistakes


Restore optimizer state after constructing the optimizer over the new
model's parameters. Do not store live `state_dict()` references in an
in-memory checkpoint. Return the saved step so the caller can continue
logging and scheduling correctly.

In [ ]:
def make_checkpoint(
    model: nn.Module, optimizer: Optimizer, step: int
) -> dict[str, Any]:
    # LLM_TODO_CH05_04_CHECKPOINT
    raise NotImplementedError("snapshot model, optimizer, and step")


def restore_checkpoint(
    checkpoint: dict[str, Any],
    model: nn.Module,
    optimizer: Optimizer,
) -> int:
    raise NotImplementedError("restore both state dictionaries")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
import copy


def train_step(model: nn.Module, optimizer: Optimizer) -> None:
    x = torch.tensor([[1.0], [2.0]])
    y = torch.tensor([[2.0], [4.0]])
    optimizer.zero_grad()
    loss = F.mse_loss(model(x), y)
    loss.backward()
    optimizer.step()


torch.manual_seed(3)
model = nn.Linear(1, 1, bias=False)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
train_step(model, optimizer)
saved = make_checkpoint(model, optimizer, step=1)

resumed = nn.Linear(1, 1, bias=False)
resumed_optimizer = torch.optim.SGD(
    resumed.parameters(), lr=0.05, momentum=0.9
)
assert restore_checkpoint(saved, resumed, resumed_optimizer) == 1
before = copy.deepcopy(resumed.weight)
train_step(resumed, resumed_optimizer)
assert not torch.equal(before, resumed.weight)
print("Exercise 5.4 passed")

## Exercise 5.5: Compare train and validation cross entropy

    **Objective.** Calculate losses on explicitly supplied toy splits and interpret them cautiously.


For one target class $y$, cross entropy is
$-\log(\operatorname{softmax}(z)_y)$. PyTorch accepts a batch of logits
`[N,C]` and integer targets `[N]`, averages the per-example losses, and
remains numerically stable by combining log-softmax with indexing.

Train/validation loss is evidence about performance on those specific
splits. Similar values do not prove that data was absent from
pretraining, while a gap alone does not identify its cause. This toy
exercise demonstrates mechanics only and deliberately avoids claims
about membership of any real corpus.


    ### Data-flow walkthrough


The training logits strongly favor their targets, while validation
logits are less decisive. Compute each loss independently and return
Python floats. The expected relation is train loss below validation
loss, not a memorized numeric constant.


    ### Hints and common mistakes


Pass raw logits to `cross_entropy`; do not apply softmax first. Ensure
target dtype is integer class indices. Keep split labels explicit so no
accidental data reuse occurs.

In [ ]:
def split_cross_entropy(
    train_logits: Tensor,
    train_targets: Tensor,
    validation_logits: Tensor,
    validation_targets: Tensor,
) -> tuple[float, float]:
    # LLM_TODO_CH05_05_CROSS_ENTROPY
    raise NotImplementedError("score both supplied splits")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
train_loss, validation_loss = split_cross_entropy(
    torch.tensor([[3.0, 0.0], [0.0, 3.0]]),
    torch.tensor([0, 1]),
    torch.tensor([[0.4, 0.6], [0.6, 0.4]]),
    torch.tensor([0, 1]),
)
assert train_loss < validation_loss
print("Exercise 5.5 passed")

## Exercise 5.6: Switch model configuration metadata

    **Objective.** Select GPT-2 variant metadata without allocating either model.


Configuration selection should be separate from model construction.
This lets tooling inspect dimensions, reject invalid names, and estimate
resources without allocating hundreds of millions of parameters.

The nominal names `124M` and `1558M` are labels used for this exercise.
Exact counts depend on conventions such as tied embeddings and bias
inclusion. Store the dimensions and nominal count as immutable metadata.


    ### Data-flow walkthrough


A lookup for `124M` should report 12 layers; `1558M` should report width
1,600, 48 layers, 25 heads, and its nominal parameter label. No
`nn.Module` is created.


    ### Hints and common mistakes


Return a clear error listing valid names. Do not silently fall back to a
smaller configuration: accidental allocation choices can be expensive.

In [ ]:
CONFIGS = {
    "124M": GPTConfig("124M", 768, 12, 12, 124_000_000),
    "1558M": GPTConfig("1558M", 1_600, 48, 25, 1_558_000_000),
}


def select_gpt2_config(name: str) -> GPTConfig:
    # LLM_TODO_CH05_06_CONFIG
    raise NotImplementedError("look up metadata without allocation")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
assert select_gpt2_config("124M").num_layers == 12
xl = select_gpt2_config("1558M")
assert (xl.embedding_dim, xl.num_layers, xl.num_heads) == (1_600, 48, 25)
print("Exercise 5.6 passed")

## Chapter 05 synthesis


- How do temperature and top-k change different parts of token selection?
- Which state is required for exact training continuation beyond this toy checkpoint?
- What can validation loss support, and what conclusions require stronger evidence?


    After answering these questions, run the chapter's `solution.py`
    from a terminal to compare behavior. The reference file is compact
    by design; the reasoning in this notebook is the main lesson.